In [1]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [2]:
CHECKPOINT_DIR = Path("notebooks/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "leaderboard_week3_best.pt"
print("Checkpoint will be saved to:", CHECKPOINT_PATH)

Checkpoint will be saved to: notebooks\models\leaderboard_week3_best.pt


In [3]:
# Standard library
import copy
import inspect
import json
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# Scikit-learn
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Neuroprobe
import neuroprobe
import neuroprobe.train_test_splits as neuroprobe_train_test_splits
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [ ]:
# =========================
# Repro / device
# =========================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Benchmark / split
# =========================
TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]

TEST_SUBJECT_ID = 1
TEST_TRIAL_ID = 2
SUBJECT_IDS = list(range(1, 11))

COORDINATE_SYSTEM = "cortical"  # lock this for all Phase 1/2 runs
USE_VAL_AS_TEST = False

# =========================
# Training
# =========================
BATCH_SIZE = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

DROPOUT = 0.1
ATTN_DROPOUT = 0.1

USE_GLOBAL_TRAIN_NORM = False

# =========================
# Signal preprocessing
# =========================
STFT_N_FFT = 64
STFT_HOP = 16
STFT_WIN_LEN = 32
LAPLACIAN_K = 4

# =========================
# Encoder dimensions
# =========================
COORD_DIM = 3
TRUNK_OUT_DIM = 96
COORD_EMB_DIM = 32
ELEC_HIDDEN_DIM = 128
MODEL_DIM = 128

SUBJ_EMB_DIM = 16
TASK_EMB_DIM = 8

# =========================
# Virtual sensor harmonizer
# =========================
NUM_VIRTUAL_SENSORS = 16  # Phase 1.5 sweep: 16 / 32 / 64
NUM_SENSOR_HEADS = 4
USE_COORDS_IN_KEYS = True
USE_COORDS_IN_VALUES = False

# =========================
# Sensor refinement
# =========================
USE_SENSOR_SELF_ATTN = True
NUM_SENSOR_SELF_ATTN_LAYERS = 1

# =========================
# Conditioning choices
# =========================
USE_SUBJECT_EMBEDDING = True
USE_TASK_EMBEDDING = True

# =========================
# Safety / debugging
# =========================
ASSERT_NONEMPTY_ELECTRODE_SET = True
RUN_PERMUTATION_TEST = True
PRINT_REAL_BATCH_CONTRACT = True

print("DEVICE:", DEVICE)
print("COORDINATE_SYSTEM:", COORDINATE_SYSTEM)

In [ ]:
subject_test = BrainTreebankSubject(
    subject_id=TEST_SUBJECT_ID,
    cache=True,
    dtype=torch.float32,
    coordinates_type=COORDINATE_SYSTEM,
)

def build_all_subjects_dict(subject_ids, test_subject, coordinate_system):
    all_subjects = {}
    test_sid = int(test_subject.subject_id)

    for sid in map(int, subject_ids):
        if sid == test_sid:
            all_subjects[sid] = test_subject
        else:
            all_subjects[sid] = BrainTreebankSubject(
                subject_id=sid,
                cache=True,
                dtype=torch.float32,
                coordinates_type=coordinate_system,
            )
    return all_subjects

ALL_SUBJECTS = build_all_subjects_dict(
    subject_ids=SUBJECT_IDS,
    test_subject=subject_test,
    coordinate_system=COORDINATE_SYSTEM,
)

In [ ]:
# =========================
# Helpers
# =========================
def task_to_id_map(tasks):
    return {t: i for i, t in enumerate(tasks)}

TASK_TO_ID = task_to_id_map(TASKS)

# =========================
# Repro
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# =========================
# Split helpers
# =========================
def _call_split_function(eval_name, split_idx=0):
    kwargs = {
        "eval_name": eval_name,
        "all_subjects": ALL_SUBJECTS,
        "test_subject_id": TEST_SUBJECT_ID,
        "test_trial_id": TEST_TRIAL_ID,
        "dtype": torch.float32,
        "lite": True,
        "nano": False,
    }

    print(f"Calling generate_splits_cross_subject with kwargs: {kwargs}")
    splits = neuroprobe_train_test_splits.generate_splits_cross_subject(**kwargs)

    fold = splits[split_idx] if isinstance(splits, list) else splits

    train_ds = fold["train_dataset"]

    if USE_VAL_AS_TEST and "val_dataset" in fold:
        eval_ds = fold["val_dataset"]
    elif "test_dataset" in fold:
        eval_ds = fold["test_dataset"]
    elif "val_dataset" in fold:
        eval_ds = fold["val_dataset"]
    else:
        raise KeyError(
            f"Expected one of 'test_dataset' or 'val_dataset' in fold keys, got: {list(fold.keys())}"
        )

    return train_ds, eval_ds, fold

def get_cross_subject_datasets(task_name, split_idx=0):
    return _call_split_function(eval_name=task_name, split_idx=split_idx)

# =========================
# Laplacian helper
# =========================
def build_laplacian_matrix(coords, k_neighbors=4):
    coords = np.asarray(coords, dtype=np.float32)
    n_electrodes = coords.shape[0]

    nn_model = NearestNeighbors(
        n_neighbors=min(k_neighbors + 1, n_electrodes),
        metric="euclidean",
    )
    nn_model.fit(coords)
    distances, indices = nn_model.kneighbors(coords)

    W = np.zeros((n_electrodes, n_electrodes), dtype=np.float32)

    for e in range(n_electrodes):
        neigh = [j for j in indices[e] if j != e]
        if len(neigh) == 0:
            continue

        neigh_d = []
        for j in neigh:
            d = np.linalg.norm(coords[e] - coords[j])
            neigh_d.append(max(d, 1e-6))
        neigh_d = np.asarray(neigh_d, dtype=np.float32)

        weights = 1.0 / neigh_d
        weights = weights / weights.sum()

        for w, j in zip(weights, neigh):
            W[e, j] = w

    return W

# =========================
# Dataset unpack helpers
# =========================
def unpack_base_item(item):
    if isinstance(item, dict):
        x = item["data"]
        y = item["label"]
        meta = item
    else:
        x, y = item[0], item[1]
        meta = {}

    if not torch.is_tensor(x):
        x = torch.tensor(x, dtype=torch.float32)
    else:
        x = x.to(torch.float32)

    y = torch.tensor(float(y), dtype=torch.float32)
    return x, y, meta

def infer_subject_id(meta):
    if "metadata" in meta and isinstance(meta["metadata"], dict):
        md = meta["metadata"]
        for key in ["subject_id", "subject_idx", "subject", "patient_id", "participant_id"]:
            if key in md:
                try:
                    return int(md[key])
                except (TypeError, ValueError):
                    pass

    for key in ["subject_id", "subject_idx", "subject", "patient_id", "participant_id"]:
        if key in meta:
            try:
                return int(meta[key])
            except (TypeError, ValueError):
                    pass

    raise KeyError(f"Could not infer subject_id from metadata keys: {list(meta.keys())}")

def infer_electrode_coordinates(meta, fallback_coords):
    if "metadata" in meta and isinstance(meta["metadata"], dict):
        md = meta["metadata"]
        for key in ["electrode_coordinates", "coords", "coordinates"]:
            if key in md:
                arr = np.asarray(md[key], dtype=np.float32)
                if arr.ndim == 2 and arr.shape[1] == 3:
                    return arr

    for key in ["electrode_coordinates", "coords", "coordinates"]:
        if key in meta:
            arr = np.asarray(meta[key], dtype=np.float32)
            if arr.ndim == 2 and arr.shape[1] == 3:
                return arr
    return fallback_coords

# =========================
# Global spectrogram stats
# =========================
@torch.no_grad() # NOTE: does not run cos USE_GLOBAL_TRAIN_NORM = False
def estimate_train_spec_stats(
    base_ds,
    fallback_coords,
    lap_k=4,
    n_fft=64,
    hop_length=16,
    win_length=32,
    max_samples=256,
):
    if not USE_GLOBAL_TRAIN_NORM:
        raise RuntimeError("estimate_train_spec_stats is disabled unless USE_GLOBAL_TRAIN_NORM is True")


In [ ]:
# =========================
# Virtual sensor harmonizer
# =========================
class VirtualSensorHarmonizer(nn.Module):
    def __init__(
            self,
            elec_hidden_dim,
            coord_dim,
            coord_emb_dim,
            model_dim,
            num_virtual_sensors,
            num_heads,
            attn_dropout=0.1,
            dropout=0.1,
            use_coords_in_keys=True,
            use_coords_in_values=False,
            use_sensor_self_attn=True,
    ):
        super().__init__()

        self.use_coords_in_keys = use_coords_in_keys
        self.use_coords_in_values = use_coords_in_values
        self.use_sensor_self_attn = use_sensor_self_attn

        self.coord_mlp = nn.Sequential(
            nn.Linear(coord_dim, coord_emb_dim),
            nn.LayerNorm(coord_emb_dim),
            nn.ReLU(),
            nn.Linear(coord_emb_dim, coord_emb_dim),
        )

        key_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_keys else 0)
        val_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_values else 0)

        self.key_proj = nn.Sequential(
            nn.Linear(key_in_dim, model_dim),
            nn.LayerNorm(model_dim),
        )
        self.val_proj = nn.Sequential(
            nn.Linear(val_in_dim, model_dim),
            nn.LayerNorm(model_dim),
        )

        self.virtual_queries = nn.Parameter(
            torch.randn(num_virtual_sensors, model_dim) * 0.02
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,
        )
        self.cross_attn_norm = nn.LayerNorm(model_dim)
        self.cross_attn_dropout = nn.Dropout(dropout)

        if use_sensor_self_attn:
            self.sensor_self_attn = nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=num_heads,
                dropout=attn_dropout,
                batch_first=True,
            )
            self.sensor_attn_norm = nn.LayerNorm(model_dim)
            self.sensor_attn_dropout = nn.Dropout(dropout)

            self.sensor_ff_norm = nn.LayerNorm(model_dim)
            self.sensor_ff = nn.Sequential(
                nn.Linear(model_dim, model_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(model_dim * 4, model_dim),
            )
            self.sensor_ff_dropout = nn.Dropout(dropout)

        self.out_norm = nn.LayerNorm(model_dim)

    def forward(self, elec_feat, coords, elec_mask):
        assert elec_feat.ndim == 3
        assert coords.ndim == 3
        assert elec_mask.ndim == 2
        assert elec_feat.shape[:2] == coords.shape[:2] == elec_mask.shape
        assert elec_mask.any(dim=1).all(), "Each sample must have at least one real electrode."

        coord_feat = self.coord_mlp(coords)

        key_in = torch.cat([elec_feat, coord_feat], dim=-1) if self.use_coords_in_keys else elec_feat
        val_in = torch.cat([elec_feat, coord_feat], dim=-1) if self.use_coords_in_values else elec_feat

        keys = self.key_proj(key_in)
        values = self.val_proj(val_in)

        B = elec_feat.size(0)
        queries = self.virtual_queries.unsqueeze(0).expand(B, -1, -1)
        key_padding_mask = ~elec_mask.bool()

        query_normed = self.cross_attn_norm(queries)
        attn_out, _ = self.cross_attn(
            query=query_normed,
            key=keys,
            value=values,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        sensors = queries + self.cross_attn_dropout(attn_out)

        if self.use_sensor_self_attn:
            sensor_normed = self.sensor_attn_norm(sensors)
            attn_out, _ = self.sensor_self_attn(
                query=sensor_normed,
                key=sensor_normed,
                value=sensor_normed,
                need_weights=False,
            )
            sensors = sensors + self.sensor_attn_dropout(attn_out)

            ff_out = self.sensor_ff(self.sensor_ff_norm(sensors))
            sensors = sensors + self.sensor_ff_dropout(ff_out)

        sensors = self.out_norm(sensors)
        return sensors

In [ ]:
# =========================
# Subject ID assumption test
# =========================
def assert_subject_ids_are_one_based(base_ds, subject_ids=SUBJECT_IDS, max_checks=64):
    observed = []
    allowed = set(map(int, subject_ids))

    n = min(len(base_ds), max_checks)
    if n == 0:
        raise ValueError("Empty dataset: cannot validate subject ID assumption.")

    sample_idxs = np.linspace(0, len(base_ds) - 1, n, dtype=int)

    for idx in sample_idxs:
        _, _, meta = unpack_base_item(base_ds[idx])
        sid = infer_subject_id(meta)
        observed.append(int(sid))

        if sid not in allowed:
            raise AssertionError(
                f"Subject ID assumption failed at dataset idx={idx}: "
                f"got subject_id={sid}, expected one of {sorted(allowed)}. "
                f"This means IDs are not matching the assumed 1-based SUBJECT_IDS space."
            )

    observed_unique = sorted(set(observed))
    print(f"Subject ID assumption check passed. Observed subject IDs: {observed_unique}")


# run on one real cross-subject training dataset
_probe_train_ds, _probe_eval_ds, _probe_fold = get_cross_subject_datasets(TASKS[0], split_idx=0)
assert_subject_ids_are_one_based(_probe_train_ds)

In [ ]:
# =========================
# Lazy dataset
# =========================
class CrossSubjectLaplacianSpectrogramDataset(Dataset):
    def __init__(
        self,
        base_ds,
        fallback_coords,
        n_fft=64,
        hop_length=16,
        win_length=32,
        lap_k=4,
        global_mean=None,
        global_std=None,
    ):
        self.base_ds = base_ds
        self.fallback_coords = np.asarray(fallback_coords, dtype=np.float32)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.lap_k = lap_k
        self.window = torch.hann_window(win_length)
        self.global_mean = global_mean
        self.global_std = global_std

    def __len__(self):
        return len(self.base_ds)

    def _laplacian_reference(self, x, coords):
        lap_w = torch.tensor(
            build_laplacian_matrix(coords, self.lap_k),
            dtype=x.dtype,
            device=x.device,
        )
        return x - lap_w @ x

    def _spectrogram(self, x_lap):
        window = self.window.to(device=x_lap.device, dtype=x_lap.dtype)

        stft = torch.stft(
            x_lap,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=window,
            center=True,
            pad_mode="reflect",
            normalized=False,
            onesided=True,
            return_complex=True,
        )

        mag = torch.log1p(torch.abs(stft))
        flat = mag.reshape(mag.shape[0], -1)

        if self.global_mean is not None and self.global_std is not None:
            if self.global_mean.shape[0] == flat.shape[0]:
                mean = self.global_mean.to(flat.device).unsqueeze(1)
                std = self.global_std.to(flat.device).unsqueeze(1).clamp_min(1e-6)
            else:
                mean = flat.mean(dim=1, keepdim=True)
                std = flat.std(dim=1, keepdim=True).clamp_min(1e-6)
        else:
            mean = flat.mean(dim=1, keepdim=True)
            std = flat.std(dim=1, keepdim=True).clamp_min(1e-6)

        return ((flat - mean) / std).reshape_as(mag)

    def __getitem__(self, idx):
        x, y, meta = unpack_base_item(self.base_ds[idx])

        subject_id = infer_subject_id(meta)
        coords = infer_electrode_coordinates(meta, self.fallback_coords)

        if coords.shape[0] != x.shape[0]:
            raise ValueError(
                f"Electrode coordinate count ({coords.shape[0]}) does not match signal electrode count ({x.shape[0]}) "
                f"for idx={idx}"
            )

        x_lap = self._laplacian_reference(x, coords)
        x_spec = self._spectrogram(x_lap)

        return {
            "x_spec": x_spec,  # [E, F, T]
            "y": y,
            "subject_idx": torch.tensor(subject_id, dtype=torch.long),
            "coords": torch.as_tensor(coords, dtype=torch.float32),  # [E, 3]
            "n_electrodes": torch.tensor(x_spec.shape[0], dtype=torch.long),
        }


def collate_cross_subject_batch(batch):
    max_e = max(item["x_spec"].shape[0] for item in batch)

    x_specs = []
    coords_list = []
    electrode_mask = []
    ys = []
    subject_idxs = []
    n_electrodes = []

    for item in batch:
        x_spec = item["x_spec"]
        coords = item["coords"]
        e, f, t = x_spec.shape
        pad_e = max_e - e

        if pad_e > 0:
            x_pad = torch.zeros((pad_e, f, t), dtype=x_spec.dtype)
            c_pad = torch.zeros((pad_e, 3), dtype=coords.dtype)
            m_pad = torch.zeros(pad_e, dtype=torch.bool)

            x_spec = torch.cat([x_spec, x_pad], dim=0)
            coords = torch.cat([coords, c_pad], dim=0)
            mask = torch.cat([torch.ones(e, dtype=torch.bool), m_pad], dim=0)
        else:
            mask = torch.ones(e, dtype=torch.bool)

        x_specs.append(x_spec)
        coords_list.append(coords)
        electrode_mask.append(mask)
        ys.append(item["y"])
        subject_idxs.append(item["subject_idx"])
        n_electrodes.append(item["n_electrodes"])

    return {
        "x_spec": torch.stack(x_specs, dim=0),           # [B, Emax, F, T]
        "coords": torch.stack(coords_list, dim=0),       # [B, Emax, 3]
        "electrode_mask": torch.stack(electrode_mask, 0),# [B, Emax]
        "y": torch.stack(ys, dim=0),
        "subject_idx": torch.tensor(subject_id - 1, dtype=torch.long),
        "n_electrodes": torch.stack(n_electrodes, dim=0),
    }

def get_fallback_coords(
    all_subjects=ALL_SUBJECTS,
    test_subject_id=TEST_SUBJECT_ID,
    trial_id: int = TEST_TRIAL_ID,
    eval_name: str = "gpt2_surprisal",
):
    base_subject = all_subjects[test_subject_id]

    ds = BrainTreebankSubjectTrialBenchmarkDataset(
        base_subject,
        trial_id=trial_id,
        dtype=torch.float32,
        eval_name=eval_name,
        lite=True,
    )

    coords = np.asarray(ds.electrode_coordinates, dtype=np.float32)
    print(
        f"Using fallback_coords from subject={test_subject_id}, "
        f"trial={trial_id}, eval_name='{eval_name}', shape={coords.shape}"
    )
    return coords
